# 01 - Live Data Ingestion

This notebook fetches live remote job feeds and persists normalized data for the rest of the project.

## Setup
Load project utilities and configure logging.

In [1]:
from __future__ import annotations

import asyncio
from pathlib import Path
import polars as pl

from job_market_copilot.config import get_settings
from job_market_copilot.logging_utils import configure_logging
from job_market_copilot.pipeline import run_ingestion

configure_logging()
settings = get_settings()
settings.ensure_directories()
print(settings.project_root)

/home/ahmad/AI/Projects/ai-job-market-copilot-ollama


## Run Ingestion
This calls Remotive API and We Work Remotely RSS feed, then normalizes and deduplicates records.

In [2]:
df_all = await run_ingestion(settings)
print(df_all.shape)
df_all.head(10)

2026-06-12 12:05:56 | INFO | job_market_copilot.pipeline | Fetching live job feeds...


2026-06-12 12:05:57 | INFO | job_market_copilot.pipeline | Normalizing and deduplicating records...


2026-06-12 12:05:57 | INFO | job_market_copilot.pipeline | Ingestion complete: 128 jobs after deduplication


(128, 14)


source_job_id,source,title,company,url,canonical_url,published_at,location,category,raw_description,salary_text,employment_type,tags,dedup_key
str,str,str,str,str,str,datetime[μs],str,str,str,str,str,list[str],str
"""2090944""","""remotive""","""Online Data Analyst Canada""","""TELUS Digital""","""https://remotive.com/remote-jo…","""https://remotive.com/remote-jo…",2026-06-04 20:58:10,"""Canada""","""All others""","""<div class=""h4"" style=""border-…","""""","""part_time""","[""go"", ""social media"", ""research""]","""https://remotive.com/remote-jo…"
"""2090945""","""remotive""","""Online Data Analyst Canada (Fr…","""TELUS Digital""","""https://remotive.com/remote-jo…","""https://remotive.com/remote-jo…",2026-06-04 21:05:36,"""Canada""","""All others""","""<p style=""border-color: rgb(22…","""""","""part_time""","[""go"", ""social media"", ""research""]","""https://remotive.com/remote-jo…"
"""2090946""","""remotive""","""Online Data Analyst United Sta…","""TELUS Digital""","""https://remotive.com/remote-jo…","""https://remotive.com/remote-jo…",2026-06-04 21:05:07,"""USA""","""All others""","""<p style=""border-color: #e5e7e…","""""","""part_time""","[""go"", ""social media"", ""research""]","""https://remotive.com/remote-jo…"
"""2090881""","""remotive""","""Business Transformation Lead""","""Expion Health""","""https://remotive.com/remote-jo…","""https://remotive.com/remote-jo…",2026-05-13 06:46:26,"""USA""","""Artificial Intelligence""","""<p><strong>Title</strong>: Bus…","""$175k - $225k""","""full_time""","[""api"", ""AI/ML"", … ""insurance""]","""https://remotive.com/remote-jo…"
"""2090887""","""remotive""","""Mid/Senior AI Cinematic Video …","""EverAI""","""https://remotive.com/remote-jo…","""https://remotive.com/remote-jo…",2026-05-19 20:34:33,"""Worldwide""","""Artificial Intelligence""","""<p class=""h1"" style=""line-heig…","""""","""full_time""","[""excel"", ""video"", … ""insurance""]","""https://remotive.com/remote-jo…"
"""2090942""","""remotive""","""Customer Operations & Writing …","""Clerky, Inc.""","""https://remotive.com/remote-jo…","""https://remotive.com/remote-jo…",2026-06-02 07:40:32,"""Worldwide""","""Customer Service""","""<p style=""min-height: 1.5em;"">…","""$12K""","""full_time""","[""AI/ML"", ""startup"", … ""testing""]","""https://remotive.com/remote-jo…"
"""2090903""","""remotive""","""Data Labeling Specialists""","""Workada""","""https://remotive.com/remote-jo…","""https://remotive.com/remote-jo…",2026-05-27 07:12:56,"""USA""","""Data and Analytics""","""<p><u>Who We Are</u><br><br>Wo…","""$18 - $22/hr""","""freelance""","[""research"", ""digital content"", ""spreadsheets""]","""https://remotive.com/remote-jo…"
"""1680495""","""remotive""","""Office Assistant""","""Coalition Technologies""","""https://remotive.com/remote-jo…","""https://remotive.com/remote-jo…",2026-05-11 20:31:07,"""Worldwide""","""Marketing""","""<p class=""h3"">WHY YOU SHOULD A…","""$31,2k- $52k""","""full_time""","[""CSS"", ""excel"", … ""Ajax""]","""https://remotive.com/remote-jo…"
"""2090900""","""remotive""","""Staff Product Engineer (Belo H…","""LawnStarter""","""https://remotive.com/remote-jo…","""https://remotive.com/remote-jo…",2026-05-27 12:27:46,"""Brazil""","""Product Management""","""<p style=""font-weight: 400;""><…","""$80k - $100k""","""full_time""","[""AWS"", ""backend"", … ""datadog""]","""https://remotive.com/remote-jo…"


## Source Coverage

In [3]:
df_all.group_by('source').len(name='count').sort('count', descending=True)

source,count
str,u32
"""wwr""",100
"""remotive""",28


## Save Quick Preview

In [4]:
preview_path = settings.resolved_artifacts_dir / 'tables' / 'normalized_preview.csv'
preview_df = df_all.head(200).with_columns(pl.col('tags').list.join(', ').alias('tags'))
preview_df.write_csv(preview_path)
preview_path

PosixPath('/home/ahmad/AI/Projects/ai-job-market-copilot-ollama/artifacts/tables/normalized_preview.csv')